# EXP-001 — serving A/B for the Duck agent (1× RTX PRO 6000, offline)

Replays the recorded Duck traffic of the 09-22 run (25 games, 1,045 requests, `arc3-serving-kit/automation/serving_pack/duck_0922.jsonl.gz`)
as **28 concurrent multi-turn agents** against each serving profile and reports **decisions per game-hour**
(valid `python` tool calls containing `action(`), tok/s, latency p50/p90, queue time, preemptions, prefix-cache hit
rate, spec-decode acceptance and tool-call validity.

Plan (`launch.py --suite exp001`, most likely winner first so a short session still answers the question):
1. `flashnext_tuned` — Flash-Next NVFP4, MTP off, 12 GiB bf16 KV, prefix caching (align), 16k ctx, 28 seqs (fallback `flashnext_nomtp_kv7_seqs20`)
2. `qwen27b_fp8` — Qwen3.8-27B-FP8 on vLLM 0.19, fp8 KV, prefix caching, MTP 2 (fallback `qwen27b_fp8_nomtp`, the proven 08-17 recipe)
3. `flashnext_baseline` — exactly the 09-22 server
4. extras while the 100-min budget lasts: `flashnext_nomtp_kv16_seqs28_pc`, `flashnext_tuned_mtp2`, `flashnext_nomtp_kv12_seqs28_pc_ctx32k`

Each profile: runtime prep → server start → wait `/v1/models` → 90 s warm-up → 12 min measured replay → teardown (never raises).
Outputs: `/kaggle/working/exp001/{exp001_summary.md, suite.json, <profile>/report.json, summary.md, server.log, requests.jsonl.gz}`.
Thesis/falsifier: `plans/EXPERIMENTS.md` EXP-001. How to read: `serving/README.md`.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

T0 = time.time()
OUT = Path("/kaggle/working/exp001")
OUT.mkdir(parents=True, exist_ok=True)

def find_kit() -> Path:
    for cand in (Path("/kaggle/input/arc3-serving-kit"), Path("/kaggle/input/datasets/kragglenote2forwork/arc3-serving-kit")):
        if (cand / "arc3-serving-kit.json").is_file():
            return cand
    for marker in Path("/kaggle/input").rglob("arc3-serving-kit.json"):
        return marker.parent
    raise FileNotFoundError("attach the private dataset kragglenote2forwork/arc3-serving-kit")

KIT = find_kit()
print("kit:", KIT, json.loads((KIT / "arc3-serving-kit.json").read_text())["built_epoch"])
print(sys.version)
subprocess.run(["nvidia-smi"], check=False)
for p in sorted(Path("/kaggle/input").glob("*")) + sorted(Path("/kaggle/input/datasets").glob("*/*")) + sorted(Path("/kaggle/input/models").glob("*/*")):
    print("input:", p)

In [ ]:
# One command: every profile in the plan, sequentially, inside a wall-clock budget. Never raises.
PLAN = os.environ.get("EXP001_PLAN", "exp001")          # or e.g. "flashnext_tuned,qwen27b_fp8,flashnext_baseline"
cmd = [sys.executable, str(KIT / "serving" / "launch.py"), "--suite", PLAN,
       "--duration", "720", "--warmup", "90", "--agents", "28", "--budget-min", "100",
       "--pack", str(KIT / "automation" / "serving_pack" / "duck_0922.jsonl.gz"),
       "--out", str(OUT), "--baseline", "flashnext_baseline"]
print(" ".join(cmd), flush=True)
env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", PYTHONUNBUFFERED="1")
try:
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
    print("suite exit code", proc.returncode, f"after {(time.time() - T0) / 60:.1f} min")
except Exception as exc:  # the notebook must finish and keep its outputs
    print("suite crashed:", repr(exc))

In [ ]:
from IPython.display import Markdown, display
summary = OUT / "exp001_summary.md"
display(Markdown(summary.read_text() if summary.exists() else "**no summary written — see suite output above**"))
for rep in sorted(OUT.glob("*/summary.md")):
    display(Markdown(rep.read_text()))
print(f"total wall clock {(time.time() - T0) / 60:.1f} min")

In [ ]:
# Keep /kaggle/working small and free of runtime caches (the GPU VM is discarded anyway).
import shutil
for p in ("/kaggle/working/vllm-site-packages",):
    shutil.rmtree(p, ignore_errors=True)
subprocess.run(["du", "-sh", str(OUT)], check=False)